# الدرس 11 - بروتوكول سياق النموذج (MCP)

بروتوكول سياق النموذج (**Model Context Protocol (MCP)**) هو معيار مفتوح يتيح للوكلاء اكتشاف واستخدام الأدوات والموارد ومصادر البيانات بشكل ديناميكي أثناء وقت التشغيل. بدلاً من ترميز الأدوات بشكل ثابت داخل الوكيل، يسمح MCP للوكلاء بالاتصال بخوادم خارجية تكشف عن القدرات عند الطلب.

في هذا الدرس، ستتعلم:
- ما هو MCP ولماذا هو مهم لأنظمة الوكلاء
- كيف يعمل هيكلية عميل-خادم MCP
- كيفية بناء وكلاء يستخدمون اكتشاف الأدوات بنمط MCP


## الإعداد

**المتطلبات المسبقة:**
- مشروع Microsoft Foundry مع نموذج منشور
- قم بتشغيل `az login` للمصادقة


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ما هو بروتوكول سياق النموذج (MCP)؟

يحدد MCP طريقة معيارية لوكلاء الذكاء الاصطناعي لاكتشاف والتفاعل مع الأدوات والمصادر الخارجية للبيانات:

- **خادم MCP**: يعرض الأدوات والموارد والتعليمات عبر بروتوكول قياسي
- **عميل MCP**: بيئة تشغيل الوكيل التي تتصل بالخوادم وتكتشف الإمكانيات المتاحة
- **الاكتشاف الديناميكي**: لا يحتاج الوكلاء إلى أدوات مشفرة ثابتا — بل يكتشفون ما هو متاح أثناء وقت التشغيل

هذا أمر قوي لبناء أنظمة وكلاء قابلة للتوسع حيث يمكن إضافة قدرات جديدة دون تعديل كود الوكيل.


## كيف يعمل MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. يتصل الوكيل (عميل MCP) بخادم MCP
2. يستجيب الخادم بقائمة الأدوات المتاحة ومخططاتها
3. يمكن للوكيل بعد ذلك استدعاء أي أداة مكتشفة أثناء تفكيره
4. تتدفق النتائج مرة أخرى عبر نفس البروتوكول


## محاكاة اكتشاف أداة MCP

بما أن خادم MCP الحقيقي يتطلب عملية خادم قيد التشغيل، سنوضح النمط باستخدام دوال `@tool` التي تحاكي ما ستوفره خدمة الإقامة المتصلة بـ MCP.

في بيئة الإنتاج، يتم اكتشاف هذه الأدوات ديناميكيًا من خادم MCP بدلاً من تعريفها محليًا.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## بناء وكيل بأدوات على نمط MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP في الإنتاج

في بيئة الإنتاج، يتيح MCP أنماطًا قوية:

- **اكتشاف الأدوات الديناميكي**: تتصل الوكلاء بخوادم MCP وتكتشف الأدوات أثناء التشغيل
- **الهندسة المعمارية المفصولة**: يمكن لمزودي الأدوات التحديث بشكل مستقل عن الوكيل
- **المشاركة عبر المنظمات**: يمكن للفرق عرض القدرات عبر خوادم MCP التي يمكن لأي وكيل استخدامها
- **دعم إطار عمل الوكيل من مايكروسوفت**: يحتوي MAF على دعم مدمج لعميل MCP عبر تكامل `mcp`

لاستخدام خادم MCP حقيقي مع MAF، ستتصل عبر `hosted_mcp_tool()` أو تكامل عميل MCP.

**تعرف على المزيد:**
- [مواصفة MCP](https://modelcontextprotocol.io/)
- [دعم MCP في إطار عمل الوكيل من مايكروسوفت](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## ملخص

في هذا الدرس، تعلمت:
- **MCP** هو معيار مفتوح لاكتشاف الأدوات الديناميكية بين الوكلاء ومزودي الأدوات
- تتيح **هيكلة الخادم-العميل** للوكلاء اكتشاف القدرات أثناء وقت التشغيل
- يتيح MCP **أنظمة وكلاء قابلة للتوسع ومنفصلة** حيث يمكن إضافة الأدوات دون تغييرات في الكود
- يوفر إطار عمل Microsoft Agent **دعم MCP مدمج** للاستخدام الإنتاجي


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
